# Review regulatory regions with the PISA squid plot

Brush **Contribution score**, optionally brush **Predicted accessibility**, then click **Annotate input** to freeze both regions. Enter a name, choose a color, add a review note, and save. Annotations appear in a separate, untitled track below the contribution scores, on the same genomic axis. Clear the brushes: your annotation remains in the chart and in the Python list `annotations`. Export CSV for complete records or BED for input intervals. Remove a saved record to correct it and save a replacement.

This is model-guided review around the Drosophila *sog* enhancer, not validation of a biological effect. Run all cells with `genome-spy-python`, `pandas`, and `ipywidgets` installed. A live Python kernel and internet for the prepared Parquet data are required. Re-running clears in-memory records; export before closing.

In [ ]:
import genome_spy as gs

DATA = "https://data.genomespy.app/datasets/bpreveal-pisa/v3/"
tracks = gs.Data(url=DATA + "fig2cd-atac-tracks.parquet", format={"type": "parquet"})
MUTED = "#d8dbe0"

# Each track updates a selection stored in the parent view's parameter scope.
output_brush = gs.selection_interval(
    "accessibilityRegion",
    push="outer",
    persist=False,
    encodings=["x"],
    extent="view",
    on="mousedown",
    clear="dblclick",
    zoom=False,
)
input_brush = gs.selection_interval(
    "contributionRegion",
    push="outer",
    persist=False,
    encodings=["x"],
    extent="view",
    on="mousedown",
    clear="dblclick",
    zoom=False,
)
hover = gs.selection_point(
    "pisaLinkHover",
    push="outer",
    persist=False,
    on="pointerover[event.shiftKey]",
    clear="mousemove[!event.shiftKey]",
)

accessibility = (
    gs.Chart(tracks)
    .transform_filter(gs.datum.track == "prediction")
    .mark_rect(minOpacity=1)
    .encode(
        x=gs.X("position:I").title("dm6 chrX"),
        y=gs.Y("value:Q").scale(zero=True).axis(None),
        y2=gs.datum(0),
        color=gs.when(output_brush)
        .then(gs.value("#332288"))
        .otherwise(gs.value(MUTED)),
        tooltip=[
            gs.Tooltip("position").title("Output position"),
            gs.Tooltip("value").title("Prediction").format(".4g"),
        ],
    )
    .properties(
        name="prediction",
        title=gs.Title(text="Predicted accessibility", style="overlay-title"),
        height={"grow": 0.16},
        cursor="text",
    )
    .add_params(output_brush)
)

# A projected brush tests the input (x) or output (x2) endpoint of a link.
# The final OR makes Shift-hover isolate links when both brushes are empty.
highlighted_link = {
    "or": [
        {"param": "pisaLinkHover", "empty": False},
        {
            "and": [
                {"param": "accessibilityRegion", "project": {"x": "x2"}},
                {"param": "contributionRegion", "project": {"x": "x"}},
                {
                    "or": [
                        {"param": "pisaLinkHover"},
                        {
                            "param": "accessibilityRegion",
                            "project": {"x": "x2"},
                            "empty": False,
                        },
                        {
                            "param": "contributionRegion",
                            "project": {"x": "x"},
                            "empty": False,
                        },
                    ]
                },
            ]
        },
    ]
}
highlight = gs.when({"ref": "highlightedLink"})
effect_color = (
    gs.Color("effect:Q")
    .scale(
        domain=[
            -0.216404,
            -0.173123,
            -0.129843,
            -0.086562,
            -0.043281,
            0,
            0.043281,
            0.086562,
            0.129843,
            0.173123,
            0.216404,
        ],
        range=[
            "#053061",
            "#2166ac",
            "#4393c3",
            "#92c5de",
            "#d1e5f0",
            "#ffffff",
            "#fddbc7",
            "#f4a582",
            "#d6604d",
            "#b2182b",
            "#67001f",
        ],
        clamp=True,
    )
    .legend(
        title="PISA (log2(fc))",
        orient="left",
        direction="vertical",
        gradientLength=84,
        gradientThickness=18,
        gradientStrokeColor="#777777",
        gradientStrokeWidth=0.5,
        values=[-0.2, 0, 0.2],
    )
)
links = (
    gs.Chart(gs.Data(url=DATA + "fig2c-atac-links.parquet", format={"type": "parquet"}))
    .transform_calculate(absEffect=gs.expr.abs(gs.datum.effect))
    .mark_link(linkShape="diagonal", orient="vertical", size=1.5, minPickingSize=2)
    .encode(
        x=gs.X("source:I").title("dm6 chrX").buildIndex(False),
        x2=gs.X2("target"),
        y=gs.datum(0, type="quantitative").scale(domain=[0, 1]).axis(None),
        y2=gs.datum(1),
        order=highlight.then(gs.value(1)).otherwise(gs.value(0)),
        color=highlight.then(effect_color).otherwise(gs.value(MUTED)),
        opacity=highlight.then(
            gs.Opacity("absEffect:Q")
            .scale(
                domain=[0, 0.216404],
                range=[0, 1],
                clamp=True,
            )
            .legend(None)
        ).otherwise(gs.value(0.1)),
        tooltip=[
            gs.Tooltip("source").title("Input position"),
            gs.Tooltip("target").title("Output position"),
            gs.Tooltip("effect").title("PISA effect (log2 fold change)").format(".4f"),
        ],
    )
    .properties(name="pisa-links", predicates={"highlightedLink": highlighted_link})
    .add_params(hover)
)

# Overlay motif intervals and labels near the bottom of the link panel.
motif_blocks = (
    gs.Chart()
    .mark_rect(minOpacity=1)
    .encode(
        color=gs.Color("motifLabel:N")
        .scale(
            domain=["M1bp", "Gaga", "Zelda"],
            range=["#bbcc33", "#44bb99", "#99ddff"],
        )
        .legend(orient="left", direction="vertical", symbolOpacity=1),
        tooltip=[
            gs.Tooltip("motifLabel").title("Motif"),
            gs.Tooltip("start").title("Start"),
            gs.Tooltip("end").title("End"),
            gs.Tooltip("strand").title("Strand"),
            gs.Tooltip("score").title("Score"),
        ],
    )
)
motif_labels = (
    gs.Chart()
    .mark_text(
        align="center",
        baseline="middle",
        paddingX=3,
        tooltip=None,
    )
    .encode(text="motifLabel", color=gs.value("black"))
)
motifs = (
    (motif_blocks + motif_labels)
    .properties(
        name="motifs",
        data=gs.Data(
            url=DATA + "fig2cd-atac-motifs.parquet",
            format={"type": "parquet"},
        ),
    )
    .transform_calculate(
        motifLabel="datum.name == 'm1bp' ? 'M1bp' : datum.name == 'gaga' ? 'Gaga' : 'Zelda'"
    )
    .encode(
        x=gs.X("start:I").title("dm6 chrX"),
        x2="end",
        y=gs.value(gs.expr("4 / height")),
        y2=gs.value(gs.expr("20 / height")),
    )
)

# Switch from compact bars to base-colored contribution letters as we zoom in.
bars = (
    gs.Chart()
    .mark_rect(minOpacity=1)
    .encode(
        color=gs.when(input_brush).then(gs.value("#332288")).otherwise(gs.value(MUTED)),
    )
)
logo = (
    gs.Chart()
    .mark_text(
        font="Source Sans Pro",
        fontWeight=700,
        size=100,
        squeeze=True,
        fitToBand=True,
        paddingX=0,
        paddingY=0,
        logoLetters=True,
    )
    .encode(
        text="base",
        color=gs.when(input_brush)
        .then(
            gs.Color("base:N")
            .scale(
                type="ordinal",
                domain=["A", "C", "G", "T", "N"],
                range=["#009E73", "#0072B2", "#F0E442", "#D55E00", "#BDBDBD"],
            )
            .legend(None)
        )
        .otherwise(gs.value(MUTED)),
    )
)
contribution = (
    gs.multiscale(
        bars,
        logo,
        stops={
            "channel": "x",
            "values": [0.15],
            "transition": {"type": "lerp", "halfLife": 60},
        },
    )
    .properties(
        name="importance",
        data=tracks,
        height={"grow": 0.16},
        cursor="text",
        title=gs.Title(text="Contribution score", style="overlay-title"),
    )
    .transform_filter(gs.datum.track == "importance")
    .encode(
        x=gs.X("position:I").title("dm6 chrX"),
        y=gs.datum(0, type="quantitative").scale(zero=True).axis(None),
        y2="value",
        tooltip=[
            gs.Tooltip("position").title("Input position"),
            gs.Tooltip("base").title("Base"),
            gs.Tooltip("value").title("Contribution").format(".4g"),
        ],
    )
    .add_params(input_brush)
)

# The same index axis uses zero-based dm6 chrX positions in every track.
saved_marks = (
    gs.Chart(data={"name": "annotations"})
    .mark_rect(opacity=0.95, cornerRadius=3)
    .encode(
        x=gs.X("start:I").title("dm6 chrX"),
        x2="end",
        color=gs.Color("color:N", scale=None).legend(None),
        tooltip=["name:N", "note:N", "start:Q", "end:Q", "follow_up:N"],
    )
)
saved_labels = saved_marks.mark_text(
    align="center", baseline="middle", paddingX=3
).encode(text="name:N", color=gs.Color("label_color:N", scale=None).legend(None))
saved_track = (saved_marks + saved_labels).properties(
    name="review-annotations", height=24
)

chart = (
    (
        accessibility
        & (links + motifs).resolve_scale(color="independent")
        & contribution
        & saved_track
    )
    .properties(
        datasets={"annotations": []},
        width=850,
        spacing=0,
        padding={"top": 8, "right": 30, "bottom": 8, "left": 10},
        scales=gs.scales(x=gs.Scale(domain=[15646649, 15647250], zoom=True)),
    )
    .add_params(
        gs.param("accessibilityRegion", value=None),
        gs.param("contributionRegion", value=None),
        gs.param("pisaLinkHover", value=None),
    )
    .resolve_scale(x="shared")
    .resolve_axis(x="shared")
    .resolve_legend(color="collected")
    .configure_axis(domain=False)
    .configure_legend(
        labelFontSize=11,
        titleFontSize=11,
        layout={"left": {"anchor": "middle", "wrap": False}},
    )
    .configure_legend_track(style=None)
    .configure_title(fontSize=12, fontWeight="normal", offset=2)
    .configure_view(stroke="transparent", strokeWidth=0)
)

In [ ]:
import asyncio
import html
import math
from datetime import datetime
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import FileLink, display

# Re-running disconnects the old widget and cancels outstanding calls.
for task_name in ("connection_task", "action_task"):
    if old_task := globals().get(task_name):
        old_task.cancel()
if old_widget := globals().get("widget"):
    old_widget.close()

widget = chart.widget(inline=True, controls=False, embed_options={"renderer": "canvas"})
annotations = []
pending = None
next_id = 1
snapshots = {}
selections = {}
action_task = None
columns = [
    "id",
    "assembly",
    "chrom",
    "start",
    "end",
    "name",
    "note",
    "color",
    "follow_up",
    "output_start",
    "output_end",
    "source",
]
name_input = widgets.Text(description="Name:", placeholder="e.g. motif-review")
note_input = widgets.Textarea(description="Note:")
color_input = widgets.ColorPicker(description="Color:", value="#d89632")
follow_input = widgets.Text(description="Follow-up:", value="Review motif perturbation")
prepare_button = widgets.Button(description="Annotate input", disabled=True)
save_button = widgets.Button(description="Save annotation", disabled=True)
cancel_button = widgets.Button(description="Cancel draft", disabled=True)
clear_button = widgets.Button(description="Clear brushes", disabled=True)
remove_choice = widgets.Dropdown(description="Saved:", options=[])
remove_button = widgets.Button(description="Remove selected", disabled=True)
csv_button = widgets.Button(description="Export CSV", disabled=True)
bed_button = widgets.Button(description="Export BED", disabled=True)
status = widgets.HTML("Connecting…")
draft_status = widgets.HTML()
table = widgets.HTML("No saved annotations yet.")
export_output = widgets.Output()


def base_interval(snapshot, lower, upper):
    """Round brush boundaries up to base edges, preserving a half-open interval."""
    bounds = snapshot.get("intervals", {}).get("x")
    if not snapshot.get("active") or not bounds:
        raise ValueError("Brush a region first.")
    if len(bounds) != 2 or not all(
        isinstance(x, (int, float)) and math.isfinite(x) for x in bounds
    ):
        raise ValueError("The brush must contain two finite positions.")
    start = max(lower, math.ceil(min(bounds)))
    end = min(upper, math.ceil(max(bounds)))
    if start >= end:
        raise ValueError("Select at least one available base.")
    return {"start": start, "end": end}


def prepare_annotation(button):
    global pending
    try:
        region = base_interval(snapshots.get("input", {}), 15646649, 15647250)
        output = snapshots.get("output", {})
        context = (
            base_interval(output, 15646499, 15647400) if output.get("active") else None
        )
        pending = {
            **region,
            "output_start": context["start"] if context else "",
            "output_end": context["end"] if context else "",
        }
        draft_status.value = f"Frozen input: chrX:{region['start']}–{region['end']} (0-based, half-open)."
        save_button.disabled = cancel_button.disabled = False
        status.value = (
            "Enter a name and note. Later brushes will not change this draft."
        )
    except ValueError as error:
        status.value = html.escape(str(error))


def cancel_draft(button=None):
    global pending
    pending = None
    save_button.disabled = cancel_button.disabled = True
    draft_status.value = ""


def refresh():
    table.value = pd.DataFrame(annotations, columns=columns).to_html(
        index=False, escape=True
    )
    remove_choice.options = [
        (f"{row['id']}: {row['name']}", row["id"]) for row in annotations
    ]
    if annotations and remove_choice.value is None:
        remove_choice.value = annotations[0]["id"]
    remove_button.disabled = csv_button.disabled = bed_button.disabled = not annotations


async def save_annotation():
    global next_id
    if pending is None:
        raise ValueError("Freeze an input region first.")
    name = name_input.value.strip()
    if not name or any(c.isspace() for c in name):
        raise ValueError("Use a nonempty name without whitespace.")
    color = color_input.value
    rgb = [int(color[i : i + 2], 16) for i in (1, 3, 5)]
    label_color = (
        "#000000"
        if sum(c * w for c, w in zip(rgb, (0.299, 0.587, 0.114))) > 150
        else "#ffffff"
    )
    record = {
        "color": color,
        "label_color": label_color,
        "id": next_id,
        "assembly": "dm6",
        "chrom": "chrX",
        **pending,
        "name": name,
        "note": note_input.value.strip(),
        "follow_up": follow_input.value.strip(),
        "source": "bpreveal-pisa/v3",
    }
    await api.datasets.set("annotations", [*annotations, record])
    annotations.append(record)
    next_id += 1
    cancel_draft()
    refresh()
    status.value = f"Saved {len(annotations)} annotation(s) in Python. Clear brushes to keep reviewing."


async def remove_annotation():
    remaining = [row for row in annotations if row["id"] != remove_choice.value]
    await api.datasets.set("annotations", remaining)
    annotations[:] = remaining
    refresh()


async def clear_brushes():
    for selection in selections.values():
        await selection.clear()


async def run_action(action):
    # Prevent duplicate saves while waiting for the browser reply.
    buttons = [prepare_button, save_button, cancel_button, remove_button, clear_button]
    for button in buttons:
        button.disabled = True
    try:
        async with asyncio.timeout(30):
            await action()
    except Exception as error:
        status.value = html.escape(f"{type(error).__name__}: {error}")
    finally:
        prepare_button.disabled = clear_button.disabled = False
        save_button.disabled = cancel_button.disabled = pending is None
        refresh()


def schedule(action):
    global action_task
    if action_task is None or action_task.done():
        action_task = asyncio.create_task(run_action(action))


def export_records(kind):
    frame = pd.DataFrame(annotations, columns=columns)
    path = Path(f"sog-review.dm6.{datetime.now():%Y%m%d-%H%M%S-%f}.{kind}")
    if kind == "bed":
        frame[["chrom", "start", "end", "name"]].to_csv(
            path, sep="\t", index=False, header=False, mode="x"
        )
    else:
        frame.to_csv(path, index=False, mode="x")
    with export_output:
        export_output.clear_output()
        display(FileLink(str(path)))


async def connect():
    global api
    try:
        async with asyncio.timeout(30):
            api = await widget.get_embed_api()
            for key, view_name, name in [
                ("input", "importance", "contributionRegion"),
                ("output", "prediction", "accessibilityRegion"),
            ]:
                view = await api.views.get({"scope": [], "view": view_name})
                selection = await view.params.get_selection(name)
                selections[key] = selection
                await selection.subscribe(
                    lambda value, key=key: snapshots.update({key: value}),
                    {"delivery": "commit"},
                )
                snapshots[key] = await selection.get_value()
        prepare_button.disabled = clear_button.disabled = False
        status.value = "Brush Contribution score, optionally brush Predicted accessibility, then Annotate input."
    except Exception as error:
        status.value = html.escape(f"Connection failed: {error}")


prepare_button.on_click(prepare_annotation)
cancel_button.on_click(cancel_draft)
for button, action in [
    (save_button, save_annotation),
    (remove_button, remove_annotation),
    (clear_button, clear_brushes),
]:
    button.on_click(
        lambda button, action=action: connection_task.get_loop().call_soon_threadsafe(
            schedule, action
        )
    )
csv_button.on_click(lambda button: export_records("csv"))
bed_button.on_click(lambda button: export_records("bed"))
display(
    widget,
    widgets.VBox(
        [
            widgets.HBox([prepare_button, clear_button]),
            draft_status,
            name_input,
            note_input,
            color_input,
            follow_input,
            widgets.HBox([save_button, cancel_button]),
            status,
            widgets.HTML("<h3>Application data — Python annotations</h3>"),
            table,
            widgets.HBox([remove_choice, remove_button]),
            widgets.HBox([csv_button, bed_button]),
            export_output,
        ]
    ),
)
connection_task = asyncio.create_task(connect())

## Use the records outside GenomeSpy

The table below is a pandas DataFrame of your own records. CSV preserves colors, notes, follow-up, optional output context, assembly, and dataset version. BED contains only input chromosome/start/end/name. All saved intervals are **dm6 chrX, 0-based and half-open**. Brush boundaries are rounded up to integer base edges and clipped to the available input/output tracks. The right edge is exclusive; no extra base is added. Pending drafts do not follow later brushes.

Python authors the chart; GenomeSpy loads prepared model results and runs declarative transforms in the browser. Python hooks own saved annotations and update the chart's named dataset. No model inference runs here.

Data: dm6 *sog* locus extracts from McAnany et al., [Positional interpretation of cis-regulatory code and nucleosome organization with deep learning models](https://doi.org/10.1038/s41467-026-74807-1), [supporting data](https://doi.org/10.5281/zenodo.20318019), prepared by the [GenomeSpy recipe](https://github.com/genome-spy/genomespy-dataset-recipes/tree/main/recipes/bpreveal-pisa). Remote extracts: GPL-2.0-or-later. Training data: GSE218852. Source motif annotations and user review notes are distinct.

In [ ]:
pd.DataFrame(annotations, columns=columns)